In [ ]:
import psycopg2
import pandas as pd
import getpass
from sqlalchemy import create_engine

DB_CONFIG = {
    "host": "192.168.100.2",        # 或 127.0.0.1
    "port": 5432,
    "database": "production_db",
    "user": "postgres",
    "password": "SkSB1660"   # 替换成你当初设置的密码
}

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
print("✅ 数据库连接成功！")

In [ ]:
# 查看当前库中的所有表以及索引
cur.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema='public' 
    ORDER BY table_name;
""")
tables = cur.fetchall()
print("📋 当前数据库中的表：")
for t in tables:
    print(f"  - {t[0]}")

cur.execute("""
    SELECT indexname 
    FROM pg_indexes 
    WHERE schemaname='public' 
    ORDER BY indexname;
""")
indexes = cur.fetchall()
print(f"\n📌 当前索引列表（共 {len(indexes)} 个）：")
for idx in indexes:
    print(f"  - {idx[0]}")

In [5]:
# 查看所有的表

# 1. 获取 public 下所有用户表（排除系统表）
cur.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public' 
      AND table_type = 'BASE TABLE'
    ORDER BY table_name;
""")
tables = cur.fetchall()

print(f"📋 共找到 {len(tables)} 个表\n")

# 2. 逐个遍历并展示
for (table_name,) in tables:
    print("=" * 80)
    print(f"📊 表名: {table_name}")
    
    # 2.1 查询总行数
    cur.execute(f'SELECT COUNT(*) FROM "{table_name}";')
    count = cur.fetchone()[0]
    print(f"📈 总行数: {count}")
    
    # 2.2 查询表结构（字段名和类型）
    cur.execute("""
        SELECT column_name, data_type 
        FROM information_schema.columns 
        WHERE table_name = %s 
        ORDER BY ordinal_position;
    """, (table_name,))
    columns_info = cur.fetchall()
    cols_str = ", ".join([f"{col} ({dtype})" for col, dtype in columns_info])
    print(f"📝 字段结构: {cols_str}")
    
    # 2.3 如果有数据，展示前 5 行
    if count > 0:
        df = pd.read_sql_query(f'SELECT * FROM "{table_name}";', conn)
        print(f"📋 示例数据 (前 5 行):")
        display(df)
    else:
        print("⚠️  该表暂无数据")
    
    print("\n")  # 空行分隔

📋 共找到 10 个表

📊 表名: battery_unit
📈 总行数: 9
📝 字段结构: ID (character varying), PO_ID (integer)
📋 示例数据 (前 5 行):


C:\Users\LiewChuanNyen\AppData\Local\Temp\ipykernel_30208\3269659500.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f'SELECT * FROM "{table_name}";', conn)


,ID,PO_ID
0,S0001,1
1,S0002,1
2,S0003,1
3,S0004,1
4,S0005,3
5,S0006,1
6,S0007,3
7,S0008,5
8,S0009,6




📊 表名: bc_grouping
📈 总行数: 0
📝 字段结构: ID (character varying), Battery_Unit_ID (character varying), Part_ID (character varying)
⚠️  该表暂无数据


📊 表名: bs_result
📈 总行数: 9
📝 字段结构: ID (integer), Employee_ID (integer), Unit_ID (character varying), Produced_Time (timestamp without time zone), Voltage (numeric), Voltage_Max (numeric), Voltage_Min (numeric), Voltage_Mean (numeric), Voltage_Range (numeric), Voltage_SD (numeric), Voltage_CV (numeric), Resistance (numeric), Resistance_Max (numeric), Resistance_Min (numeric), Resistance_Mean (numeric), Resistance_Range (numeric), Resistance_SD (numeric), Resistance_CV (numeric)
📋 示例数据 (前 5 行):


C:\Users\LiewChuanNyen\AppData\Local\Temp\ipykernel_30208\3269659500.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f'SELECT * FROM "{table_name}";', conn)


...

C:\Users\LiewChuanNyen\AppData\Local\Temp\ipykernel_30208\3269659500.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f'SELECT * FROM "{table_name}";', conn)


,ID,Product_ID,WO_ID,Total,Remaining
0,4,BATTERY,WO0002,6,6
1,7,UPSBUBC4,WO0004,22,0
2,8,BATTERY,WO0004,4,4
3,1,UPSBUBC4,WO0001,10,5
4,3,UPSBUBC4,WO0002,5,3
5,5,UPSBUBC4,WO0003,3,0
6,6,BATTERY,WO0003,6,0




📊 表名: wd_result
📈 总行数: 0
📝 字段结构: ID (integer), Employee_ID (integer), Unit_ID (character varying), Produced_Time (timestamp without time zone), Welding_Axis_X (numeric), Welding_Axis_Y (numeric), Welding_Axis_Z (numeric), Welding_Axis_R (numeric), Welding_Time (numeric), Peak_Voltage (numeric), Average_Voltage (numeric), Peak_Current (numeric), Average_Current (numeric), Power (numeric), Resistance (numeric)
⚠️  该表暂无数据


📊 表名: work_order
📈 总行数: 4
📝 字段结构: ID (character varying), Status (character varying)
📋 示例数据 (前 5 行):


C:\Users\LiewChuanNyen\AppData\Local\Temp\ipykernel_30208\3269659500.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f'SELECT * FROM "{table_name}";', conn)


,ID,Status
0,WO0001,On-going
1,WO0002,On-going
2,WO0003,On-going
3,WO0004,On-going


In [ ]:
# Input Employee details
name = input("Employee name:")
bs_access = True if input("BS Access (True/False):").lower() in ['1', 'true', 't', 'yes', 'y'] else False
pc_access = True if input("PC Access (True/False):").lower() in ['1', 'true', 't', 'yes', 'y'] else False
wd_access = True if input("WD Access (True/False):").lower() in ['1', 'true', 't', 'yes', 'y'] else False
cur.execute('INSERT INTO "employee" ("Name", "BS_Access", "PC_Access", "WD_Access") VALUES (%s, %s, %s, %s) RETURNING "ID";', (name, bs_access, pc_access, wd_access))
conn.commit()  # 提交事务
new_id = cur.fetchone()[0]
print(f"✅ New employee inserted, automatically generated ID is: {new_id}")

In [ ]:
# Input Product details
id = input("Product ID:")
cell_count = int(input("Cell Count:"))
voltage_upper_limit = float(input("Voltage Upper Limit:"))
voltage_lower_limit = float(input("Voltage Lower Limit:"))
resistance_upper_limit = float(input("Resistance Upper Limit:"))
resistance_lower_limit = float(input("Resistance Lower Limit:"))
holding_duration = float(input("Holding Duration:"))
voltage_range_threshold = float(input("Voltage Range Threshold:"))
voltage_sd_threshold = float(input("Voltage SD Threshold:"))
voltage_cov_threshold = float(input("Voltage CoV Threshold:"))
resistance_range_threshold = float(input("Resistance Range Threshold:"))
resistance_sd_threshold = float(input("Resistance SD Threshold:"))
resistance_cov_threshold = float(input("Resistance CoV Threshold:"))
cur.execute('INSERT INTO "product" ("ID", "Cell_Count", "Voltage_Upper_Limit", "Voltage_Lower_Limit", "Resistance_Upper_Limit", "Resistance_Lower_Limit", "Holding_Duration", "Voltage_Range_Threshold", "Voltage_SD_Threshold", "Voltage_COV_Threshold", "Resistance_Range_Threshold", "Resistance_SD_Threshold", "Resistance_COV_Threshold") VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s) RETURNING "ID";', (id, cell_count, voltage_upper_limit, voltage_lower_limit, resistance_upper_limit, resistance_lower_limit, holding_duration, voltage_range_threshold, voltage_sd_threshold, voltage_cov_threshold, resistance_range_threshold, resistance_sd_threshold, resistance_cov_threshold))
conn.commit()
print(f"✅ New product inserted.")

In [ ]:
# Input Work Order details
name = input("Work Order ID:")
status = input("Status:")
cur.execute('INSERT INTO "work_order" ("ID", "Status") VALUES (%s, %s) RETURNING "ID";', (name, status))
conn.commit()
new_id = cur.fetchone()[0]
print(f"✅ New work order inserted.")

In [ ]:
# Input Product Order details
product_id = input("Product ID:")
work_order_id = input("Work Order ID:")
total = input("Total:")
remaining = input("Remaining:")
cur.execute('INSERT INTO "product_order" ("Product_ID", "WO_ID", "Total", "Remaining") VALUES (%s, %s, %s, %s) RETURNING "ID";', (product_id, work_order_id, total, remaining))
conn.commit()
new_id = cur.fetchone()[0]
print(f"✅ New product order inserted.")

In [ ]:
conn.rollback()

In [ ]:
cur.close()
conn.close()